# Fig. 2 · Travel time and spatial alignment

All panels use the supplied 10,115 residential H3 cells, prefiltered for complete amenity coverage, valid recurrent and nearest-set travel times, non-zero nearest-set travel time, and 0 < T_prox < 50 min.
The 40+ group pools the original 35–40 minute group with all longer times.
Panels d–g use equal H3 means within each 1 km grid for both $q_K$ and $\Delta T_{rel}$. In panel a, the histogram stops at 40 minutes and the last point is placed at 40 to mark the pooled tail.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'figure_style.py').is_file())
sys.path.insert(0, str(root / 'src'))
from figure_style import add_panel_label, configure_style, export_figure, figure_size

DATA_DIR = root / 'data'
configure_style('final')


## a–c. Travel time by accessibility

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aggregate_cells import aggregate_to_1km

TIME_BINS = [str(edge) for edge in range(5, 40, 5)] + ['40+']


def proximity_groups(cells):
    """Attach the common T_prox groups used in Figures 2 and 3."""
    result = cells.copy()
    result['T_prox_bin'] = pd.cut(
        result.T_prox, [*range(0, 36, 5), np.inf], labels=TIME_BINS, right=False,
    )
    return result


def summarise_proximity(cells, metrics):
    """Calculate group sizes, mean T_prox and each metric's mean and quartiles."""
    grouped = cells.groupby('T_prox_bin', observed=True)
    summary = grouped.agg(n=('T_prox', 'size'), x=('T_prox', 'mean'))
    for metric in metrics:
        summary[metric + '_mean'] = grouped[metric].mean()
        summary[metric + '_q25'] = grouped[metric].quantile(.25)
        summary[metric + '_q75'] = grouped[metric].quantile(.75)
    return summary.reset_index()

REGIONS = ['Helsinki', 'Tampere', 'Turku', 'Oulu']
time = pd.read_parquet(DATA_DIR / 'aggregated' / 'travel_time_h3.parquet')
time['DeltaT'] = time.T_freq - time.T_prox
time['DeltaT_ratio'] = time.DeltaT / time.T_prox
cells = proximity_groups(time)
metrics = ['T_freq', 'DeltaT', 'DeltaT_ratio']
summary = pd.concat([
    summarise_proximity(cells, metrics).assign(series='Pooled'),
    *(summarise_proximity(cells.loc[cells.region.eq(region)], metrics).assign(series=region)
      for region in REGIONS),
], ignore_index=True)
pooled = summary.loc[summary.series.eq('Pooled')].set_index('T_prox_bin').reindex(TIME_BINS)
colours = ['#0052CC', '#E69F00', '#009E73', '#CC79A7']

def quartiles(ax, x, data, metric, colour, alpha=.6):
    ax.errorbar(x, (data[metric + '_q25'] + data[metric + '_q75']) / 2,
                yerr=(data[metric + '_q75'] - data[metric + '_q25']) / 2,
                fmt='none', color=colour, alpha=alpha, linewidth=.6, capsize=1.7)

def draw_travel_time(a):
    hist = a.twinx()
    counts, _, _ = hist.hist(cells.T_prox, bins=32, range=(0, 40),
                             color='#aaaaaa', alpha=.25)
    hist.set(yscale='log', ylim=(1, max(counts.max()**4, 2)))
    hist.set_axis_off()
    a.set_zorder(hist.get_zorder() + 1)
    a.patch.set_visible(False)
    for region, colour in zip(REGIONS, colours):
        rows = summary.loc[summary.series.eq(region)].set_index('T_prox_bin').reindex(TIME_BINS)
        keep = rows.n.ge(30)
        x = rows.x.mask(rows.index == '40+', 40).where(keep)
        a.plot(x, rows.T_freq_mean.where(keep), color=colour,
               alpha=.45, marker='o', ms=2.5, label=region)
        quartiles(a, x, rows.where(keep), 'T_freq', colour, .35)
    x = pooled.x.mask(pooled.index == '40+', 40)
    a.plot(x, pooled.T_freq_mean, 'o-', color='#333333', ms=2.6, label='All', zorder=5)
    quartiles(a, x, pooled, 'T_freq', '#333333')
    a.axhline(15, color='#888888', linestyle='--', linewidth=.65)
    a.plot([0, 40], [0, 40], ':', color='#888888', linewidth=.65)
    a.set(xlim=(0, 41), ylim=(0, 90), xlabel=r'$T_{prox}$ (min)', ylabel=r'$T_{freq}$ (min)')
    a.set_xticks(range(0, 5 * (len(TIME_BINS) + 1), 5), ['0', *TIME_BINS])
    a.legend(frameon=False, loc='upper left')



def draw_time_difference(ax, metric, ylabel, colour):
    x = np.arange(len(TIME_BINS))
    ax.bar(x, pooled[metric + '_mean'], width=.52, color=colour, alpha=.8)
    quartiles(ax, x, pooled, metric, colour)
    ax.axhline(0, color='#222222', linewidth=.55)
    ax.set(xlabel=r'$T_{prox}$ (min)', ylabel=ylabel, xlim=(-.65, len(x) - .35))
    ax.set_xticks(x, TIME_BINS, rotation=45, ha='right')


fig, ax = plt.subplots(figsize=(75 / 25.4, 75 / 25.4))
fig.subplots_adjust(left=.16, right=.97, bottom=.15, top=.95)
draw_travel_time(ax)
add_panel_label(ax, 'a', x=-.16)
export_figure(fig, 'fig2/fig2a-travel-time')

for label, metric, ylabel, colour, stem in [
    ('b', 'DeltaT', r'$\Delta T$ (min)', '#211E55', 'absolute-difference'),
    ('c', 'DeltaT_ratio', r'$\Delta T_{rel}$', '#E74C22', 'relative-difference'),
]:
    fig, ax = plt.subplots(figsize=(75 / 25.4, 48 / 25.4))
    draw_time_difference(ax, metric, ylabel, colour)
    add_panel_label(ax, label, x=-.16)
    export_figure(fig, f'fig2/fig2{label}-{stem}')

## d–g. Bivariate maps

The maps use OpenStreetMap tiles at low opacity. `FIG2_BASEMAP=none` disables the basemap.

In [ ]:
import os

import contextily as ctx
import geopandas as gpd
import matplotlib as mpl
from matplotlib.colors import BoundaryNorm, ListedColormap
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from shapely.affinity import scale

empirical = pd.read_parquet(DATA_DIR / 'aggregated' / 'empirical_qk_h3.parquet')
matching = gpd.read_parquet(DATA_DIR / 'aggregated' / 'h3_grid_matching.parquet')
qk_grid = aggregate_to_1km(
    empirical.assign(qk_empirical_count=empirical.n_users),
    matching, ['qk_empirical'],
)
grid_time = (time.merge(matching[['home_gid9', 'grd_id']], on='home_gid9', validate='one_to_one')
             .groupby('grd_id', as_index=False)['DeltaT_ratio'].mean())
geography = (matching.loc[matching.map_region.notna()]
             .drop_duplicates('grd_id').sort_values('grd_id')
             [['grd_id', 'geometry', 'sum', 'vaesto', 'map_region']]
             .rename(columns={'map_region': 'region'}))
maps = (geography.merge(qk_grid[['grd_id', 'qk_empirical', 'n_users']], on='grd_id')
        .merge(grid_time, on='grd_id'))
maps = maps.loc[maps.vaesto.gt(100) & maps.n_users.gt(5)].copy()
for metric in ['qk_empirical', 'DeltaT_ratio']:
    maps[metric + '_bin'] = maps.groupby('region')[metric].transform(
        lambda values: pd.cut(values, [-np.inf, *values.quantile([1/3, 2/3]), np.inf],
                              labels=False, include_lowest=True),
    )
maps['bivar_class'] = 3 * maps.qk_empirical_bin + maps.DeltaT_ratio_bin
use_basemap = os.environ.get('FIG2_BASEMAP') != 'none'
if use_basemap:
    ctx.tile.USER_AGENT = 'K-Visitation-Reproduction/1.0'

palette = {
    (0, 0): "#bbaad3", (1, 0): "#e2e2e2", (2, 0): "#ffc575",
    (0, 1): "#705ca1", (1, 1): "#b7b7b7", (2, 1): "#f14f22",
    (0, 2): "#210f53", (1, 2): "#747474", (2, 2): "#aa0000",
}
colors = [palette[(code // 3, code % 3)] for code in range(9)]
cmap = ListedColormap(colors)
class_norm = BoundaryNorm(np.arange(-0.5, 9.5, 1), ncolors=9)


def set_region_view(ax, data, region):
    quantiles = [.02, .98] if region in ('Tampere', 'Oulu') else [0, 1]
    bounds = data.geometry.centroid.get_coordinates().quantile(quantiles).to_numpy()
    centre = bounds.mean(axis=0)
    span = bounds[1] - bounds[0]
    if region == 'Oulu':
        span *= [1.75, .72]
    half_span = np.maximum(span / 2, 1000)
    if region == 'Turku':
        centre[1] -= .2 * half_span[1]
    lower, upper = centre - 1.03 * half_span, centre + 1.03 * half_span
    ax.set(xlim=(lower[0], upper[0]), ylim=(lower[1], upper[1]))


def scaled_by_poi(data):
    result = data.copy()
    bins = np.arange(0, 5.5, 0.5)
    indices = np.clip(np.digitize(np.log1p(result["sum"].fillna(0)), bins, right=True), 0, len(bins) - 1)
    factors = np.linspace(0.55, 1.25, len(bins))[indices]
    result["geometry"] = [
        scale(geometry, xfact=float(factor), yfact=float(factor), origin="centroid")
        for geometry, factor in zip(result.geometry, factors)
    ]
    return result


def draw_map(ax, region):
    data = maps.loc[maps.region.eq(region)]
    set_region_view(ax, data, region)
    ax.set_facecolor("#f5f5f5")
    if use_basemap:
        ctx.add_basemap(
            ax, crs=data.crs, source=ctx.providers.OpenStreetMap.Mapnik,
            alpha=.35, attribution=False, zorder=0,
        )
    scaled_by_poi(data).plot(
        column="bivar_class", cmap=cmap, norm=class_norm,
        linewidth=0.28, edgecolor="white", alpha=0.94, zorder=5, ax=ax,
    )
    scale_bar = AnchoredSizeBar(
        ax.transData, 5_000, "5 km", loc="lower left", pad=0.25,
        borderpad=0.55, sep=2, size_vertical=120, color="#222222", frameon=True,
        fontproperties=mpl.font_manager.FontProperties(family="Arial", size=5),
    )
    scale_bar.patch.set_facecolor("white")
    scale_bar.patch.set_edgecolor("none")
    scale_bar.patch.set_alpha(0.80)
    scale_bar.set_zorder(20)
    ax.add_artist(scale_bar)
    if use_basemap:
        ax.text(.99, .01, '© OpenStreetMap contributors', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=5, zorder=25,
                bbox=dict(facecolor='white', edgecolor='none', alpha=.85, pad=1))
    ax.set_axis_off()

In [ ]:
for label, region in zip('defg', REGIONS):
    fig, ax = plt.subplots(figsize=(75 / 25.4, 57 / 25.4))
    draw_map(ax, region)
    ax.set_title(region)
    add_panel_label(ax, label, x=-.04)
    export_figure(fig, f'fig2/fig2{label}-{region.lower()}')

fig, legend = plt.subplots(figsize=(30 / 25.4, 30 / 25.4))
for q in range(3):
    for dt in range(3):
        legend.add_patch(plt.Rectangle((q, dt), 1, 1, facecolor=palette[q, dt],
                                       edgecolor='white', linewidth=.6))
legend.set(xlim=(0, 3), ylim=(0, 3), aspect='equal', xticks=[], yticks=[],
           xlabel=r'$q_K$ low → high', ylabel=r'$\Delta T_{rel}$ low → high')
for spine in legend.spines.values():
    spine.set_visible(False)
export_figure(fig, 'fig2/fig2-map-legend')